In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns
from matplotlib.lines import Line2D
from plotting_defaults import COLOR_MAP, delete_tex_cache, save_and_measure, set_style

from fedaugment.utils import cd_to_git_root

cd_to_git_root()
delete_tex_cache()
set_style()

_ = pl.Config.set_tbl_cols(-1)
_ = pl.Config.set_tbl_rows(30)
_ = pl.Config.set_tbl_width_chars(200)
_ = pl.Config.set_float_precision(3)

plots_dir = Path("analysis/plots/efficiency")
plots_dir.mkdir(exist_ok=True, parents=True)

2026-04-22 17:26:02.051 | DEBUG    | fedaugment.utils.misc:cd_to_git_root:154 - Changed working directory to: /home/lbhm/Projects/fedaugment


## Inference Analysis

In [2]:
inference_base_path = Path("logs/efficiency")

results_by_exp_group: dict[str, pl.DataFrame] = {}
for exp_group_path in inference_base_path.iterdir():
    if not exp_group_path.is_dir():
        continue

    df_list = [
        pl.read_csv(ckpt_path).with_columns(pl.lit(ckpt_path.name.split("-")[1]).alias("method"))
        for ckpt_path in exp_group_path.iterdir()
    ]

    results_by_exp_group[exp_group_path.name] = pl.concat(df_list)

In [3]:
aggregated: dict[str, pl.DataFrame] = {}

for exp_group in ["12_views"]:
    aggregated[exp_group] = (
        results_by_exp_group[exp_group]
        .group_by(["method", "batch_size"])
        .agg(
            [
                pl.mean("inference_time_ms").alias("mean_inference_time_ms"),
                pl.std("inference_time_ms").alias("std_inference_time_ms"),
                pl.min("inference_time_ms").alias("min_inference_time_ms"),
                pl.max("inference_time_ms").alias("max_inference_time_ms"),
                pl.mean("peak_gpu_memory_gb"),
            ]
        )
        .with_columns(
            ((pl.col("batch_size") * 1000) / (pl.col("mean_inference_time_ms"))).alias(
                "mean_throughput_qps"
            )
        )
        .sort(["method", "batch_size"])
    )
batch_sizes = (
    aggregated["12_views"].select(pl.col("batch_size")).unique().to_series().sort().to_list()
)

In [4]:
for exp_group in ["12_views"]:
    # Max width: 241.14 pt
    fig, ax = plt.subplots(1, 1, figsize=(3.525, 1.7), layout="tight")

    df = aggregated[exp_group].filter(
        pl.col("method").is_in(["cl_optim", "la2m_default", "la2m_nopca", "v2v"])
    )
    ax.vlines(
        x=32, ymin=1e3, ymax=1e7, colors=COLOR_MAP["centralized"], linestyles="--", alpha=0.8
    )
    sns.lineplot(
        data=df, x="batch_size", y="mean_throughput_qps", hue="method", palette=COLOR_MAP, ax=ax
    )

    ax.set_xlabel("Batch size")
    ax.set_ylabel("Throughput (queries/sec)")
    ax.set_xlim(min(batch_sizes), max(batch_sizes))
    ax.set_ylim(1e3, 3.5 * 1e6)
    ax.set_yscale("log")
    ax.set_xscale("log")

    ax.legend(
        handles=[
            Line2D([0], [0], color=COLOR_MAP["cl_optim"], label=r"\system{}"),
            Line2D([0], [0], color=COLOR_MAP["la2m_default"], label="LA2M"),
            Line2D([0], [0], color=COLOR_MAP["la2m_nopca"], label="LA2M (no PCA)"),
            Line2D([0], [0], color=COLOR_MAP["v2v"], label="vec2vec"),
        ],
        loc="upper center",
        ncol=4,
        bbox_to_anchor=(0.5, 1.15),
    )
    _ = save_and_measure(fig, plots_dir / f"{exp_group}-inference_throughput.pdf")

analysis/plots/efficiency/12_views-inference_throughput.pdf: 239.725x105.479 pt (8.46x3.72 cm)


In [5]:
for exp_group in ["12_views"]:
    # Max width: 241.14 pt
    fig, ax = plt.subplots(1, 1, figsize=(3.525, 1.7), layout="tight")

    df = aggregated[exp_group].filter(
        pl.col("method").is_in(["cl_optim", "la2m_default", "la2m_nopca", "v2v"])
    )
    sns.lineplot(
        data=df, x="batch_size", y="mean_inference_time_ms", hue="method", palette=COLOR_MAP, ax=ax
    )

    for method in df["method"].unique():
        method_data = df.filter(pl.col("method") == method)
        ax.fill_between(
            method_data["batch_size"],
            method_data["mean_inference_time_ms"] - method_data["std_inference_time_ms"],
            method_data["mean_inference_time_ms"] + method_data["std_inference_time_ms"],
            alpha=0.3,
            color=COLOR_MAP[method],
        )

    ax.set_xlabel("Batch size")
    ax.set_ylabel("Inference time (ms)")
    ax.set_xlim(min(batch_sizes) - 1, max(batch_sizes) + 1)
    ax.set_xticks([1, *ax.get_xticks()[1:-1]])

    ax.legend(
        handles=[
            Line2D([0], [0], color=COLOR_MAP["cl_optim"], label=r"\system{}"),
            Line2D([0], [0], color=COLOR_MAP["la2m_default"], label="LA2M"),
            Line2D([0], [0], color=COLOR_MAP["la2m_nopca"], label="LA2M (no PCA)"),
            Line2D([0], [0], color=COLOR_MAP["v2v"], label="vec2vec"),
        ],
        loc="upper center",
        ncol=4,
        bbox_to_anchor=(0.5, 1.15),
    )
    _ = save_and_measure(fig, plots_dir / f"{exp_group}-inference_time.pdf")

analysis/plots/efficiency/12_views-inference_time.pdf: 239.760x105.570 pt (8.46x3.72 cm)


In [6]:
aggregated["12_views"].filter(pl.col("batch_size") == 32)

method,batch_size,mean_inference_time_ms,std_inference_time_ms,min_inference_time_ms,max_inference_time_ms,peak_gpu_memory_gb,mean_throughput_qps
str,i64,f64,f64,f64,f64,f64,f64
"""cl_default""",32,0.485,0.078,0.441,1.314,0.338,66030.180
"""cl_noval""",32,0.229,0.046,0.209,0.727,0.186,139939.442
"""cl_optim""",32,0.226,0.047,0.208,0.738,0.186,141421.952
"""la2m_default""",32,3.603,1.046,0.296,4.558,0.215,8880.483
"""la2m_nopca""",32,3.298,1.020,0.005,4.320,17.124,9702.535
"""union_minus""",32,0.010,0.001,0.008,0.014,0.009,3267020.197
"""union_plus""",32,0.028,0.002,0.022,0.037,0.010,1160073.532
"""v2v""",32,0.793,0.016,0.755,0.846,1.044,40358.892


## Training Analysis

Relevant training logs:
- `v2v`: 20260114_220126
- `cl_default`, `cl_optim`, `la2m_default`, `la2m_nopca`: 20260114_220123
- `cl_noval`: 20260116_101754

In [7]:
train_base_path = Path("logs/projection-training")

df_list = [pl.read_csv(train_run / "summary.csv") for train_run in train_base_path.iterdir()]

training_results = pl.concat(df_list)
print(f"Read in {len(training_results)} training runs.")

Read in 6 training runs.


In [8]:
training_results[
    ["model_name", "num_views", "training_time", "train_max_gpu_memory_gb"]
].with_columns((pl.col("training_time") / 3600).alias("training_time_hours")).sort(["model_name"])

model_name,num_views,training_time,train_max_gpu_memory_gb,training_time_hours
str,i64,f64,f64,f64
"""cl_default""",12,23301.200,50.869,6.473
"""cl_noval""",12,12854.890,49.614,3.571
"""cl_optim""",12,21321.940,50.593,5.923
"""la2m_default""",12,2389.180,52.163,0.664
"""la2m_nopca""",12,15895.610,65.299,4.415
"""v2v""",12,86435.180,37.218,24.010
